# Crop Yield Capstone – Team Notebook (11 Members, 11 Sections)
**Machine Learning and AI (Essentials) · Thrive Campus · Group project**  |  Crop: **Rice, paddy** (change `CROP` in Section 2 if your group picks another)

You are the data science team for a national agriculture ministry. The ministry asks three questions: **How much will we harvest?** (regression) · **Will this be a bad year?** (classification) · **What kinds of growing years do we have?** (clustering).

This notebook splits the whole capstone into **11 sections, one per team member**. Each section has `TODO` and `HINT` comments telling you what to write. A finished solution exists separately (`Crop_Yield_Capstone_SOLUTION.ipynb`) for the teaching team – don't ask for it; learn by doing.

## Team roster (fill in)
| Section | Member | Task | Marks area | GitHub username |
|---|---|---|---|---|
| 1 | Member 1 | Set up and load data | Part 1 | |
| 2 | Member 2 | Filter crop + 3 observations | Part 1 | |
| 3 | Member 3 | Clean the data + cleaning log | Cleaning (15) | |
| 4 | Member 4 | Charts 1 and 2 | Charts (15) | |
| 5 | Member 5 | Charts 3 and 4 | Charts (15) | |
| 6 | Member 6 | Regression: features + split | Regression (20) | |
| 7 | Member 7 | Regression: two models, scores, coefficients | Regression (20) | |
| 8 | Member 8 | Classification: target, leakage, split | Classification (20) | |
| 9 | Member 9 | Classification: models, metrics, confusion matrix | Classification (20) | |
| 10 | Member 10 | Clustering | Clustering (10) | |
| 11 | Member 11 | Grid Search tuning + final summary | Tuning (5) | |

Report (15 marks) and presentation: **everyone** contributes – each member writes up their own section for the report and presents it.

## How the team works together
1. **Sections run top to bottom and depend on each other.** Each section lists *Receives* (variables you can use from earlier sections) and *Hands over* (variables the next person needs). Keep those variable names exactly.
2. **Work on your own branch**, e.g. `section-07-regression`, and open a pull request. Everyone commits – no commits means no group mark.
3. **Weekly checkpoints** (see the milestone table in the brief): Members 1–2 → Milestones 1–2; 3 → M3; 4–5 → M4; 6–7 → M5; 8–9 → M6; 10–11 → M7; everyone → M8 (report and slides). Missing two checkpoints in a row is how groups fail.
4. **Until the person before you has pushed**, use the fall-back lines in each section to test your code.
5. Before the final hand-in, one person runs **Kernel → Restart & Run All**. An error at any point costs −5.

## Rules that run through everything
* **Every model result ends with a sentence starting “This means the ministry should…”.** Technical work without a recommendation scores zero for that section.
* **Honest numbers only.** A poor result explained well beats an inflated one. R² above 0.95 → probably scored on training data. Accuracy of 1.00 → leakage.
* **Comment every code line you write** so a teammate can explain it. AI assistants are allowed for explanations – note where you used one in the table at the very bottom.
* Penalties: notebook errors −5 · regression scored on training data −7 · yield column in classification inputs −10.

---
# SECTION 1 · Part 1 – Set up and load the data
**Assigned to:** Member 1 (Albert Kwamibuor)  |  **Marks area:** Part 1 (set-up, milestone 1)

**Receives:** nothing (you are first)
**Hands over:** `raw` (the full DataFrame), `RANDOM_STATE`, `save_fig()`, `RESULTS`, all imports

**Goal:** make the notebook run on anyone's computer and load `yield_df.csv`.

**Step 1.1 – Imports and settings.**

In [1]:
import os, json, warnings                      # os: folders, json: save results, warnings: keep output tidy
import numpy as np                             # numerical arrays and maths
import pandas as pd                            # tables (DataFrames)
import matplotlib.pyplot as plt                # drawing charts
import seaborn as sns                          # nicer chart styling

from sklearn.model_selection import train_test_split, GridSearchCV      # splitting data / tuning
from sklearn.preprocessing import StandardScaler                        # put features on the same scale
from sklearn.pipeline import Pipeline                                   # chain scaler + model safely
from sklearn.linear_model import LinearRegression, LogisticRegression   # simple, explainable models
from sklearn.ensemble import RandomForestRegressor, RandomForestClassifier  # flexible tree-based models
from sklearn.dummy import DummyClassifier                               # "always guess the majority" baseline
from sklearn.cluster import KMeans                                      # clustering algorithm
from sklearn.metrics import (mean_squared_error, r2_score,              # regression metrics
                             accuracy_score, precision_score, recall_score, f1_score,  # classification metrics
                             confusion_matrix, ConfusionMatrixDisplay, silhouette_score)

warnings.filterwarnings("ignore")              # hide harmless warnings so the notebook output stays readable
sns.set_theme(style="whitegrid", context="notebook")   # one consistent chart style
RANDOM_STATE = 42                              # fixed seed -> everyone gets the same numbers every run
os.makedirs("figures", exist_ok=True)          # charts are saved here (used by the report and slides)
os.makedirs("results", exist_ok=True)          # key numbers are saved here
RESULTS = {}                                   # dictionary we fill with headline numbers as we go

def save_fig(fig, name):
    """Save a chart into the figures/ folder at good quality."""
    fig.savefig(f"figures/{name}.png", dpi=160, bbox_inches="tight")

**Step 1.2 – Load the file.**

In [2]:
raw = pd.read_csv("yield_df.csv")              # FileNotFoundError here = the CSV is not in the notebook's folder
print("Raw shape (rows, columns):", raw.shape)
raw.head()

Raw shape (rows, columns): (28242, 8)


,Unnamed: 0,Area,Item,Year,hg/ha_yield,average_rain_fall_mm_per_year,pesticides_tonnes,avg_temp
0,0,Albania,Maize,1990,36613,1485.0,121.0,16.37
1,1,Albania,Potatoes,1990,66667,1485.0,121.0,16.37
2,2,Albania,"Rice, paddy",1990,23333,1485.0,121.0,16.37
3,3,Albania,Sorghum,1990,12500,1485.0,121.0,16.37
4,4,Albania,Soybeans,1990,7000,1485.0,121.0,16.37


**Step 1.3 – Write 2–3 sentences:** what does one row represent? Which column is the leftover index? How many crops and countries are there?

In [3]:
raw.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 28242 entries, 0 to 28241
Data columns (total 8 columns):
 #   Column                         Non-Null Count  Dtype  
---  ------                         --------------  -----  
 0   Unnamed: 0                     28242 non-null  int64  
 1   Area                           28242 non-null  object 
 2   Item                           28242 non-null  object 
 3   Year                           28242 non-null  int64  
 4   hg/ha_yield                    28242 non-null  int64  
 5   average_rain_fall_mm_per_year  28242 non-null  float64
 6   pesticides_tonnes              28242 non-null  float64
 7   avg_temp                       28242 non-null  float64
dtypes: float64(3), int64(3), object(2)
memory usage: 1.7+ MB


---
# SECTION 2 · Part 1 (continued) – Filter to our crop and write three observations
**Assigned to:** Member 2 (write your name here: ________)  |  **Marks area:** Part 1 (milestones 1–2): three observations written

**Receives:** `raw`
**Hands over:** `df` (only your crop's rows, still un-cleaned)

**Goal:** narrow the data to one staple crop and notice things *before* cleaning.

**Step 2.1 – Look at the crop names.**

In [ ]:
# TODO: print how many rows each crop has.
# HINT: raw["Item"].value_counts().  Names must match EXACTLY: "Rice, paddy" has a comma; "Rice" alone matches nothing.
# YOUR CODE HERE

**Step 2.2 – Filter.**

In [ ]:
# TODO: set CROP = "Rice, paddy" (or your group's crop) and create df = raw[raw["Item"] == CROP].copy()
# TODO: add an assert that len(df) > 0 so a spelling mistake fails loudly.
# TODO: print the number of rows, countries (df['Area'].nunique()) and the first/last year.
# YOUR CODE HERE

**Step 2.3 – Gather evidence for your observations.**

In [ ]:
# TODO: print df.describe() for the four numeric columns.
# TODO: count how many rows each (Area, Year) pair has.  HINT: df.groupby(["Area","Year"]).size().value_counts()
#       If it is not all 1s, the data is NOT one-row-per-country-per-year - Member 3 needs to know!
# YOUR CODE HERE

**Step 2.4 – Three observations.** Replace the numbered lines below with your own, each containing a specific number.

In [ ]:
# (No code needed - write in the markdown cell that follows.)

**Observations (edit me):**
1. TODO
2. TODO
3. TODO

---
# SECTION 3 · Part 2 – Clean the data (and fill the cleaning log)
**Assigned to:** Member 3 (write your name here: ________)  |  **Marks area:** Cleaning (15 marks) – milestone 3

**Receives:** `df`
**Hands over:** `rice` (clean, ONE row per country-year, columns renamed: yield_hg_ha, rain_mm, pesticides_t, temp_c), `cleaning_df` (the log)

**Goal:** every cleaning decision has a written reason: types, duplicates, missing values, outliers.

**Step 3.1 – Start a cleaning log.**

In [ ]:
# TODO: create an empty list cleaning_log and a helper log(step, found, decision, reason) that appends a dict to it.
# HINT: cleaning_log.append({"Step": step, "What we found": found, "What we did": decision, "Why": reason})
# YOUR CODE HERE

**Step 3.2 – Index column and names.**

In [ ]:
# TODO: drop the 'Unnamed: 0' column and rename the long columns:
#   hg/ha_yield -> yield_hg_ha, average_rain_fall_mm_per_year -> rain_mm, pesticides_tonnes -> pesticides_t, avg_temp -> temp_c
# HINT: df.drop(columns=[...]) and df.rename(columns={...}).   Then call log(...) to record what you did and WHY.
# YOUR CODE HERE

**Step 3.3 – Data types.**

In [ ]:
# TODO: print df.dtypes. Convert every numeric column with pd.to_numeric(df[c], errors="coerce").
# HINT: "could not convert string to float" means a number is stored as text.  Record the result in the log.
# YOUR CODE HERE

**Step 3.4 – Missing values.**

In [ ]:
# TODO: count missing values per column (df.isna().sum()), then decide: drop rows, fill, or nothing needed?  Log the decision.
# HINT: "Input contains NaN" later in the project means this step was incomplete.
# YOUR CODE HERE

**Step 3.5 – Exact duplicates.**

In [ ]:
# TODO: count and remove rows that are identical in every column.  HINT: df.duplicated().sum(), df.drop_duplicates().  Log it.
# YOUR CODE HERE

**Step 3.6 – The repeated country-years (important!).**

In [ ]:
# TODO: for each (Area, Year), count distinct values per column.  HINT: df.groupby(["Area","Year"])[cols].nunique()
#       Which column(s) disagree between repeated rows?
# TODO: decide how to get ONE row per country-year (HINT: groupby(["Area","Year"], as_index=False).agg(...), averaging the column that disagrees
#       and taking the first value of the rest).  Save the result as `rice`.
# TODO: prove it worked with assert not rice.duplicated(["Area","Year"]).any()
# TODO: log your decision, including how many rows were removed and WHY (think: what would happen to a train/test split otherwise?)
# YOUR CODE HERE

**Step 3.7 – Ranges and outliers.**

In [ ]:
# TODO: print rice.describe(). Check yields > 0, rainfall > 0, pesticides >= 0 (use assert).
# TODO: count outliers in yield with the IQR rule (below Q1-1.5*IQR or above Q3+1.5*IQR). Decide keep or remove, and log WHY.
# TODO: check: does rain_mm take more than one value inside a country?  HINT: rice.groupby("Area")["rain_mm"].nunique().max()
#       (Members 6 and 8 need the answer!)
# YOUR CODE HERE

**Step 3.8 – Show the log and save the clean data.**

In [ ]:
# TODO: turn cleaning_log into a DataFrame `cleaning_df`, display it, and save results/cleaning_log.csv and results/rice_clean.csv
# TODO: print how many rows you started with and ended with.
# YOUR CODE HERE

### ✍️ Your written answer for Section 3 (double-click to edit)
> **This means the ministry should…** *(TODO: finish this sentence in plain English, using the numbers you found above. Maximum 3 sentences.)*

---
# SECTION 4 · Part 3 – Explore with charts (charts 1 and 2)
**Assigned to:** Member 4 (write your name here: ________)  |  **Marks area:** Charts (15 marks) – milestone 4 (shared with Member 5)

**Receives:** `rice`, `save_fig`
**Hands over:** charts saved in `figures/`

**Goal:** two charts, each with a title, labelled axes and a *specific* written interpretation (mention numbers; say what it means).

**Chart 1 – distribution of yield.**

In [ ]:
# TODO: histogram of rice["yield_hg_ha"] with a vertical line at the median.
# HINT: ax.hist(..., bins=40); ax.axvline(median, linestyle="--")
# TODO: title, x label WITH UNITS, y label, legend.  Save with save_fig(fig, "chart1_yield_distribution").
# YOUR CODE HERE

**Interpretation of chart 1** (edit): What shape is it? What is the median? What does the spread imply for the ministry?

> TODO

**Chart 2 – yield over time.**

In [ ]:
# TODO: compute the mean yield per Year and draw a line chart. HINT: rice.groupby("Year")["yield_hg_ha"].mean()
# TODO: fit a straight line with np.polyfit(years, means, 1) and print the gain in hg/ha per year.
# TODO: title, axis labels, save_fig(fig, "chart2_yield_trend").
# YOUR CODE HERE

### ✍️ Your written answer for Section 4 (double-click to edit)
> **This means the ministry should…** *(TODO: finish this sentence in plain English, using the numbers you found above. Maximum 3 sentences.)*

**Interpretation of chart 2 (edit):** Is the trend up or down? By how much? Why are there bumps?

> TODO

---
# SECTION 5 · Part 3 (continued) – Explore with charts (charts 3 and 4)
**Assigned to:** Member 5 (write your name here: ________)  |  **Marks area:** Charts (15 marks) – milestone 4

**Receives:** `rice`, `save_fig`
**Hands over:** charts saved in `figures/`; a correlation table

**Goal:** two more charts with the same standard as Section 4. Between you and Member 4 you must cover four DIFFERENT charts.

**Chart 3 – temperature vs yield.**

In [ ]:
# TODO: scatter plot of temp_c (x) vs yield_hg_ha (y) with a trend line. HINT: alpha=0.3 makes overlapping points readable.
# TODO: print the correlation: rice["temp_c"].corr(rice["yield_hg_ha"])
# TODO: title, axis labels with units, save_fig(fig, "chart3_temp_vs_yield")
# YOUR CODE HERE

**Interpretation of chart 3:** TODO (direction, strength, and one WARNING about what the chart does NOT prove - hint: is the temperature difference mostly between countries or within them?)

**Chart 4 – pesticides vs yield.**

In [ ]:
# TODO: scatter plot of pesticides_t vs yield_hg_ha. Pesticide tonnes range from 0 to hundreds of thousands, so try a log x-axis
# HINT: ax.set_xscale("log") and plot rice["pesticides_t"] + 1 (so zeros can be drawn)
# TODO: print a rank correlation: rice["pesticides_t"].corr(rice["yield_hg_ha"], method="spearman")
# TODO: title, labels, save_fig(fig, "chart4_pesticides_vs_yield")
# YOUR CODE HERE

**Interpretation of chart 4:** TODO (does more pesticide go with more yield? Is that cause-and-effect?)

**Correlation table.**

In [ ]:
# TODO: show correlations of yield with rain_mm, pesticides_t, temp_c, Year (HINT: rice[[...]].corr()["yield_hg_ha"]) and store in RESULTS["corr_with_yield"].
# YOUR CODE HERE

### ✍️ Your written answer for Section 5 (double-click to edit)
> **This means the ministry should…** *(TODO: finish this sentence in plain English, using the numbers you found above. Maximum 3 sentences.)*

---
# SECTION 6 · Part 4 – Regression (a): choose inputs, encode, split
**Assigned to:** Member 6 (write your name here: ________)  |  **Marks area:** Regression (20 marks) – milestone 5 (shared with Member 7)

**Receives:** `rice`, `RANDOM_STATE`, `RESULTS`
**Hands over:** `X`, `y`, `X_train`, `X_test`, `y_train`, `y_test`, `baseline_rmse`, `num_features`

**Goal:** build the input table X and target y and split correctly. **Scoring on training data later costs −7**, so get the split right.

**Step 6.1 – Choose inputs.**

In [ ]:
# TODO: set TARGET = "yield_hg_ha" and num_features = ["Year", "pesticides_t", "temp_c"].
# THINK: rain_mm is ONE value per country (Member 3 checked). Why would including both rain_mm and a country label confuse a linear model?
#        Write your answer as a comment here: ...
# TODO: one-hot encode the country column and combine with the numeric features.
# HINT: pd.get_dummies(rice[["Area"] + num_features], columns=["Area"], drop_first=True, dtype=float)
# TODO: y = rice[TARGET]; print X.shape
# YOUR CODE HERE

**Step 6.2 – Split.**

In [ ]:
# TODO: split into 80% train / 20% test. HINT: train_test_split(X, y, test_size=0.2, random_state=RANDOM_STATE)
# TODO: print the sizes of the train and test sets.
# YOUR CODE HERE

**Step 6.3 – Baseline.**

In [ ]:
# TODO: compute the error of a 'no skill' model that always predicts the TRAINING mean yield, measured on the TEST set (RMSE).
# HINT: np.sqrt(mean_squared_error(y_test, np.full(len(y_test), y_train.mean())))  -> store in baseline_rmse and RESULTS["baseline_rmse"]
# YOUR CODE HERE

**Step 6.4 – Explain (3 sentences):** why do we split BEFORE fitting anything, and why must the final scores come from the test set?

> TODO

---
# SECTION 7 · Part 4 – Regression (b): two models, honest scores, coefficients in real units
**Assigned to:** Member 7 (write your name here: ________)  |  **Marks area:** Regression (20 marks) – milestone 5

**Receives:** `X_train, X_test, y_train, y_test, baseline_rmse, rice`
**Hands over:** `lin`, `rf` (fitted models), `reg_table`, `X`, `y` used again by Member 11

**Goal:** compare two models on the TEST set and read what they mean.

**Step 7.1 – Fit two models on TRAINING data only.**

In [ ]:
# TODO: fit LinearRegression() and RandomForestRegressor(n_estimators=300, random_state=RANDOM_STATE, n_jobs=-1).
# HINT: model.fit(X_train, y_train)
# YOUR CODE HERE

**Step 7.2 – Score both models on train AND test.**

In [ ]:
# TODO: for each model compute RMSE (np.sqrt(mean_squared_error(...))) and R2 (r2_score(...)) on BOTH sets; collect into a DataFrame `reg_table`.
# WARNING: R2 above 0.95 on the TEST set means something is wrong (training data scored? duplicates leaking?).
# TODO: compare to baseline_rmse.  Store results in RESULTS["reg"].
# YOUR CODE HERE

**Step 7.3 – Say it in plain English.**

In [ ]:
# TODO: print one sentence per model such as: "typically off by about ___ hg/ha (__% of the average yield) and explains __% of the variation".
# TODO: print Train R2 minus Test R2 for each model. Which one over-fits more?
# YOUR CODE HERE

**Step 7.4 – Coefficients in real units (linear model).**

In [ ]:
# TODO: print the Linear Regression coefficients for Year, pesticides_t and temp_c as sentences, e.g. "each extra year: +__ hg/ha".
# HINT: pd.Series(lin.coef_, index=X.columns).  Scale to a meaningful size (e.g. per +10,000 tonnes of pesticide).
# TODO (stretch): fit a second linear model WITHOUT country labels but WITH rain_mm and report the rainfall coefficient per +100 mm.
#       Why can't you put rain_mm and the country labels in the same linear model?
# YOUR CODE HERE

**Step 7.5 – What does the forest rely on?**

In [ ]:
# TODO: get rf.feature_importances_, add up the importance of all 'Area_' columns into one 'Country' bar and chart it with Year, pesticides, temperature.
# YOUR CODE HERE

**Step 7.6 – Actual vs predicted chart (test set).**

In [ ]:
# TODO: scatter of y_test (x) vs rf.predict(X_test) (y) with a dashed y = x line. Title, labels, save_fig(fig, "reg_actual_vs_pred").
# YOUR CODE HERE

**Step 7.7 – A harder, more honest test: forecast the future.**

In [ ]:
# TODO: train on years <= 2008 and test on years 2009-2013 (use a boolean mask on rice["Year"]); report RMSE and R2 for both models.
# Why is this score lower than the random-split score, and which one should the ministry trust for planning imports?
# YOUR CODE HERE

### ✍️ Your written answer for Section 7 (double-click to edit)
> **This means the ministry should…** *(TODO: finish this sentence in plain English, using the numbers you found above. Maximum 3 sentences.)*

---
# SECTION 8 · Part 5 – Classification (a): define 'bad year', avoid leakage, split
**Assigned to:** Member 8 (write your name here: ________)  |  **Marks area:** Classification (20 marks) – milestone 6 (shared with Member 9)

**Receives:** `rice`, `RANDOM_STATE`
**Hands over:** `idx_train`, `idx_test`, `Xc_train`, `Xc_test`, `yc_train`, `yc_test`, `clf_features`

**Goal:** create the yes/no target and choose inputs so the model cannot read the answer. **A yield column among the inputs costs −10.**

**Step 8.1 – Split first.**

In [ ]:
# TODO: split the row INDEX of rice 80/20: idx_train, idx_test = train_test_split(rice.index, test_size=0.2, random_state=RANDOM_STATE)
# YOUR CODE HERE

**Step 8.2 – Define a bad year.**

In [ ]:
# TODO: decide what 'bad' means. Suggested rule: yield more than 15% below that COUNTRY's own typical (median) yield.
# WHY per country? (write your reason as a comment) ...
# TODO: compute each country's median yield using TRAINING rows only: rice.loc[idx_train].groupby("Area")["yield_hg_ha"].median()
# TODO: create rice["typical_yield"] (map the medians onto every row) and rice["bad_year"] = (yield < 0.85 * typical).astype(int)
# TODO: check for countries with no training rows (typical_yield is NaN) and exclude them. Print the % of bad years in train and test.
# YOUR CODE HERE

**Step 8.3 – Choose inputs WITHOUT leakage.**

In [ ]:
# TODO: set clf_features = ["Year", "rain_mm", "pesticides_t", "temp_c"].
# TODO: write an assert that fails if any input name contains 'yield', or equals 'typical_yield' or 'bad_year'.
# THINK: why is 'typical_yield' also leakage even though it does not say 'bad'? (comment your answer)
# TODO: build Xc_train, Xc_test, yc_train, yc_test from the indices.
# YOUR CODE HERE

**Step 8.4 – Explain (3 sentences):** why do we define 'bad' relative to each country rather than with one national cut-off?

> TODO

---
# SECTION 9 · Part 5 – Classification (b): models, accuracy / precision / recall, confusion matrix
**Assigned to:** Member 9 (write your name here: ________)  |  **Marks area:** Classification (20 marks) – milestone 6

**Receives:** `Xc_train, Xc_test, yc_train, yc_test`
**Hands over:** `clf_table`, `preds`, `best_name`

**Goal:** build a warning model and explain its mistakes in plain English.

**Step 9.1 – Models.**

In [ ]:
# TODO: create three models in a dict: DummyClassifier(strategy="most_frequent") as a baseline; a Pipeline of StandardScaler + LogisticRegression(max_iter=1000, class_weight="balanced");
#       RandomForestClassifier(n_estimators=300, class_weight="balanced", random_state=RANDOM_STATE).
# HINT: ConvergenceWarning means scale first and raise max_iter.
# TODO: fit each on the TRAINING data and predict the TEST data.
# YOUR CODE HERE

**Step 9.2 – Metrics table.**

In [ ]:
# TODO: for each model compute accuracy, precision, recall (and F1) on the TEST set; collect into a DataFrame `clf_table`.
# HINT: precision_score(y_test, pred, zero_division=0)
# WARNING: accuracy of 1.00 = leakage. Go back to Section 8.
# YOUR CODE HERE

**Step 9.3 – Confusion matrices.**

In [ ]:
# TODO: draw a confusion matrix for each real model. HINT: ConfusionMatrixDisplay(confusion_matrix(y_test, pred), display_labels=[...]).plot(ax=ax)
# TODO: save_fig(fig, "clf_confusion")
# YOUR CODE HERE

**Step 9.4 – Explain the mistakes in plain English.**

In [ ]:
# TODO: pick the best real model (justify: which matters more to the ministry, missing a bad year or a false alarm?).
# TODO: unpack tn, fp, fn, tp = confusion_matrix(...).ravel() and print one sentence for EACH of the four numbers
#       (e.g. "30 bad years looked normal to the model -> farmers get no early support").
# TODO: compare accuracy with the 'always normal' baseline. Why is accuracy misleading when bad years are rare?
# YOUR CODE HERE

### ✍️ Your written answer for Section 9 (double-click to edit)
> **This means the ministry should…** *(TODO: finish this sentence in plain English, using the numbers you found above. Maximum 3 sentences.)*

---
# SECTION 10 · Part 6 – Clustering: what kinds of growing years do we have?
**Assigned to:** Member 10 (write your name here: ________)  |  **Marks area:** Clustering (10 marks) – milestone 7

**Receives:** `rice`, `RANDOM_STATE`, `RESULTS`
**Hands over:** `rice['cluster']`, `rice['growing_type']`, `summary` table

**Goal:** scale, choose k with evidence, fit K-Means and give every group a plain-English name.

**Step 10.1 – Scale first.**

In [ ]:
# TODO: choose cluster columns (e.g. yield_hg_ha, rain_mm, pesticides_t, temp_c) and scale them with StandardScaler().fit_transform(...)
# WHY? (comment). If every point lands in one cluster you forgot this step.
# YOUR CODE HERE

**Step 10.2 – Choose k with evidence.**

In [ ]:
# TODO: for k in 2..8 fit KMeans(n_clusters=k, n_init=10, random_state=RANDOM_STATE); record inertia_ and silhouette_score(Z, labels_).
# TODO: draw an elbow plot and a silhouette plot, save_fig(fig, "cluster_choose_k").
# TODO: also look at the cluster SIZES for k=3,4,5. A tiny cluster (<3% of rows) is usually not a useful 'type'.
# TODO: write the justification of your k in a markdown cell below.
# YOUR CODE HERE

**Step 10.3 – Fit, profile, name.**

In [ ]:
# TODO: fit the final KMeans and add rice["cluster"].
# TODO: profile each cluster: mean of each column, number of rows, number of countries. HINT: rice.groupby("cluster")[cols].mean()
# TODO: give EVERY cluster a plain-English name that a minister understands (e.g. 'Cool, high-input, high-yield'), stored in rice["growing_type"].
# HINT: print example countries per cluster: rice[rice.cluster==c]["Area"].value_counts().head(5)
# YOUR CODE HERE

**Step 10.4 – Visualise and interpret.**

In [ ]:
# TODO: scatter plot (e.g. temp_c vs yield) coloured by growing_type, with title, labels, legend. save_fig(fig, "cluster_scatter")
# TODO: check how many countries stay in ONE cluster in every year: rice.groupby("Area")["cluster"].nunique() == 1
#       What does that say about what drives the groups - weather from year to year, or the farming system of each country?
# YOUR CODE HERE

### ✍️ Your written answer for Section 10 (double-click to edit)
> **This means the ministry should…** *(TODO: finish this sentence in plain English, using the numbers you found above. Maximum 3 sentences.)*

**Why this k? (edit):** TODO – use the elbow, the silhouette scores AND the cluster sizes.

---
# SECTION 11 · Part 7 – Tune with Grid Search, final summary, and hand-in check
**Assigned to:** Member 11 (write your name here: ________)  |  **Marks area:** Tuning (5 marks) – milestone 7, plus final integration

**Receives:** `X_train, X_test, y_train, y_test`, `rf`, `RESULTS`, `clf_table`, `best_name`, `K`, `silhouette`
**Hands over:** `results/summary.json`

**Goal:** run Grid Search on your best regression model, report before vs after *honestly*, and assemble the final scoreboard.

**Step 11.1 – Grid Search.**

In [ ]:
# TODO: define a small param_grid for the Random Forest, e.g. {"n_estimators": [200, 400], "max_depth": [None, 20], "min_samples_leaf": [1, 2, 4]}
# TODO: run GridSearchCV(RandomForestRegressor(random_state=RANDOM_STATE), param_grid, cv=5, scoring="neg_root_mean_squared_error").fit(X_train, y_train)
# Only the TRAINING data may be used during tuning. Print grid.best_params_ and the cross-validated RMSE (remember the minus sign!).
# YOUR CODE HERE

**Step 11.2 – Before vs after, honestly.**

In [ ]:
# TODO: score the original `rf` and the tuned grid.best_estimator_ on the TEST set (RMSE and R2) and show them side by side.
# TODO: calculate the % change. If tuning did NOT help, say so - that is a valid, marked result.
# YOUR CODE HERE

**Step 11.3 – Final scoreboard.**

In [ ]:
# TODO: build a 3-row table: Question | Method | Headline test result, for regression, classification and clustering.
# TODO: save RESULTS to results/summary.json with json.dump(..., default=float)
# YOUR CODE HERE

**Step 11.4 – Hand-in check (tick each box by editing this cell).**

- [ ] Kernel → Restart & Run All finishes with no error
- [ ] `yield_df.csv` is in the repository folder and README explains how to run
- [ ] Every section ends with 'This means the ministry should…'
- [ ] Every member has commits
- [ ] No yield column among classification inputs; regression scored on TEST data
- [ ] AI use noted in the table below

In [ ]:
# (No code - tick the boxes above.)

### ✍️ Your written answer for Section 11 (double-click to edit)
> **This means the ministry should…** *(TODO: finish this sentence in plain English, using the numbers you found above. Maximum 3 sentences.)*

---
# Wrap-up (whole team)
### AI-assistant use log (honesty table)
| Section | Member | What you asked the AI for | What you changed or verified yourself |
|---|---|---|---|
| | | | |

### Final recommendations to the ministry (write together after all sections are finished)
1. **Forecasting:** This means the ministry should… TODO
2. **Early warning:** This means the ministry should… TODO
3. **Planning by type:** This means the ministry should… TODO